# Task 3 — Predictive Customer Churn Classifier
## Random Forest & XGBoost | Datheon Machine Learning & Data Science Internship

**Goal:** predict telecom customer churn, account for class imbalance, compare Random Forest and XGBoost with stratified cross-validation, and report ROC-AUC, a confusion matrix, and top-10 feature importance.

**Dataset:** IBM Telco Customer Churn, 7,043 rows and 21 original columns. The notebook downloads the CSV from IBM's public GitHub repository. Internet access is required for the first data load.

**Important:** the requested ROC-AUC > 0.80 is a target, not a result to hard-code. All scores are calculated from the held-out test set.


## 0. Environment setup
Run this notebook in Jupyter or Google Colab. If needed, run the following in a notebook cell before continuing:

`%pip install -r requirements.txt`

If you are using Colab and have not uploaded the requirements file, use:
`%pip install -q numpy pandas matplotlib seaborn scikit-learn xgboost`


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay,
    classification_report, accuracy_score, precision_score, recall_score, f1_score
)

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")
DATA_URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
print("Setup complete.")


## 1. Load and validate the IBM dataset
`TotalCharges` is stored as text in the source. Blank strings are converted to missing numeric values and handled in the preprocessing pipeline. `customerID` is removed because it is an identifier, not a useful behavioral predictor.


In [ ]:
df = pd.read_csv(DATA_URL)
print("Dataset shape:", df.shape)
print("Expected original shape: approximately (7043, 21)")
display(df.head())

required_columns = {"customerID", "TotalCharges", "Churn"}
missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(
        f"The downloaded CSV is not the expected IBM Telco dataset. Missing columns: {sorted(missing_columns)}. "
        "Check DATA_URL and use the IBM raw CSV URL shown in the setup cell."
    )

# Clean data types without discarding the 11 zero-tenure customers with blank charges.
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"].astype(str).str.strip().replace("", np.nan), errors="coerce"
)
df = df.drop(columns=["customerID"])
df = df.dropna(subset=["Churn"]).copy()
print("Rows after basic validation:", len(df))
print("Missing values in TotalCharges after conversion:", int(df["TotalCharges"].isna().sum()))
print("Target labels:", df["Churn"].value_counts(dropna=False).to_dict())


## 2. Explore the target and a useful numeric feature
The dataset is imbalanced: fewer customers churn than stay. Accuracy alone is not sufficient, so the model comparison includes ROC-AUC, precision, recall, and F1.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=df, x="Churn", ax=axes[0])
axes[0].set_title("Customer counts by churn label")
sns.boxplot(data=df, x="Churn", y="tenure", ax=axes[1])
axes[1].set_title("Tenure by churn label")
plt.tight_layout()
plt.show()

display(df["Churn"].value_counts().rename_axis("Churn").to_frame("count"))


## 3. Train/test split and preprocessing
Split the data before fitting preprocessors. Numeric values are median-imputed; categorical values are most-frequent-imputed and one-hot encoded. `handle_unknown="ignore"` prevents errors when a category appears only in validation/test data. Class imbalance is addressed with model class weights.


In [ ]:
X = df.drop(columns="Churn")
y = df["Churn"].map({"No": 0, "Yes": 1})
if y.isna().any():
    raise ValueError("Unexpected labels in Churn. Expected only 'Yes' and 'No'.")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=["number"]).columns.tolist()

preprocessor = ColumnTransformer(transformers=[
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), numeric_features),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_features)
])

print("Training rows:", len(X_train), "| Test rows:", len(X_test))
print("Numeric columns:", numeric_features)
print("Categorical column count:", len(categorical_features))
print("Training churn rate:", round(float(y_train.mean()), 4))


## 4. Random Forest — stratified cross-validation and tuning
Grid search uses ROC-AUC as the selection metric. Class weighting is applied during model training to account for the minority churn class.


In [ ]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

rf_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", RandomForestClassifier(
        random_state=RANDOM_STATE, class_weight="balanced", n_jobs=1
    ))
])

rf_search = GridSearchCV(
    estimator=rf_pipeline,
    param_grid={
        "model__n_estimators": [200, 350],
        "model__max_depth": [None, 12],
        "model__min_samples_leaf": [1, 3]
    },
    scoring="roc_auc", cv=cv, n_jobs=-1, refit=True, verbose=1
)
rf_search.fit(X_train, y_train)
print("Random Forest best mean CV ROC-AUC:", round(float(rf_search.best_score_), 4))
print("Random Forest best parameters:", rf_search.best_params_)


## 5. XGBoost — stratified cross-validation and tuning
XGBoost is a required comparison model for this task. Install it with `pip install xgboost` if the import fails. `scale_pos_weight` gives additional weight to the minority churn class.


In [ ]:
try:
    from xgboost import XGBClassifier
except ImportError as exc:
    raise ImportError(
        "XGBoost is required for Task 3 comparison. Install it with `%pip install xgboost`, "
        "restart the kernel if prompted, then rerun all cells."
    ) from exc

positive_count = int((y_train == 1).sum())
negative_count = int((y_train == 0).sum())
scale_pos_weight = negative_count / max(positive_count, 1)

xgb_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", XGBClassifier(
        objective="binary:logistic", eval_metric="logloss",
        n_estimators=250, learning_rate=0.05, max_depth=4,
        subsample=0.9, colsample_bytree=0.9,
        scale_pos_weight=scale_pos_weight,
        random_state=RANDOM_STATE, n_jobs=1, tree_method="hist"
    ))
])

xgb_search = GridSearchCV(
    estimator=xgb_pipeline,
    param_grid={
        "model__n_estimators": [200, 300],
        "model__max_depth": [3, 5]
    },
    scoring="roc_auc", cv=cv, n_jobs=1, refit=True, verbose=1
)
xgb_search.fit(X_train, y_train)
print("XGBoost best mean CV ROC-AUC:", round(float(xgb_search.best_score_), 4))
print("XGBoost best parameters:", xgb_search.best_params_)


## 6. Evaluate both models on the held-out test set
The test set is not used for hyperparameter selection. ROC-AUC uses predicted probabilities, not class labels. A 0.50 threshold is used for the initial classification metrics.


In [ ]:
models = {
    "Random Forest": rf_search.best_estimator_,
    "XGBoost": xgb_search.best_estimator_
}

metric_rows = []
probabilities = {}
predictions = {}
for name, model in models.items():
    prob = model.predict_proba(X_test)[:, 1]
    pred = (prob >= 0.50).astype(int)
    probabilities[name] = prob
    predictions[name] = pred
    metric_rows.append({
        "Model": name,
        "CV ROC-AUC": rf_search.best_score_ if name == "Random Forest" else xgb_search.best_score_,
        "Test ROC-AUC": roc_auc_score(y_test, prob),
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0)
    })

metrics = pd.DataFrame(metric_rows).sort_values("Test ROC-AUC", ascending=False).reset_index(drop=True)
display(metrics.round(4))
best_model_name = str(metrics.loc[0, "Model"])
print("Best model by held-out test ROC-AUC:", best_model_name)
print("ROC-AUC > 0.80 target reached:", bool(metrics.loc[0, "Test ROC-AUC"] > 0.80))
print("\nClassification report at threshold 0.50 —", best_model_name)
print(classification_report(
    y_test, predictions[best_model_name], target_names=["No churn", "Churn"], zero_division=0
))


## 7. ROC curve comparison
The ROC curve visualizes the true-positive versus false-positive rate across thresholds. The diagonal line represents random ranking.


In [ ]:
plt.figure(figsize=(7, 6))
for name, prob in probabilities.items():
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc_value = roc_auc_score(y_test, prob)
    plt.plot(fpr, tpr, label=f"{name} (AUC={auc_value:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", label="Random baseline")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve — Customer Churn")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()


## 8. Confusion matrix for the best test ROC-AUC model
The confusion matrix shows correct and incorrect predictions at a 0.50 probability threshold.


In [ ]:
best_model = models[best_model_name]
best_pred = predictions[best_model_name]
cm = confusion_matrix(y_test, best_pred, labels=[0, 1])
display(pd.DataFrame(cm, index=["Actual No churn", "Actual Churn"], columns=["Predicted No churn", "Predicted Churn"]))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No churn", "Churn"]).plot(values_format="d")
plt.title(f"Confusion Matrix — {best_model_name}")
plt.tight_layout()
plt.show()


## 9. Top 10 feature importances
For both models, feature importance is extracted after fitting the preprocessing pipeline. These values indicate model reliance, not causation.


In [ ]:
fitted_preprocessor = best_model.named_steps["preprocess"]
fitted_classifier = best_model.named_steps["model"]
feature_names = fitted_preprocessor.get_feature_names_out()
importances = fitted_classifier.feature_importances_
if len(feature_names) != len(importances):
    raise ValueError("Feature-name and importance lengths do not match.")
importance_df = pd.DataFrame({"feature": feature_names, "importance": importances})
importance_df = importance_df.sort_values("importance", ascending=False).head(10).reset_index(drop=True)
display(importance_df)

plt.figure(figsize=(9, 5))
sns.barplot(data=importance_df, x="importance", y="feature")
plt.title(f"Top 10 Feature Importances — {best_model_name}")
plt.xlabel("Importance")
plt.ylabel("Encoded feature")
plt.tight_layout()
plt.show()


## 10. Threshold sensitivity (diagnostic only)
This test-set table illustrates the precision/recall trade-off. For a production decision, choose a threshold on a validation set using business costs and campaign capacity, then evaluate once on a separate test set.


In [ ]:
best_prob = probabilities[best_model_name]
threshold_rows = []
for threshold in [0.30, 0.40, 0.50, 0.60, 0.70]:
    pred = (best_prob >= threshold).astype(int)
    threshold_rows.append({
        "threshold": threshold,
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),
        "f1": f1_score(y_test, pred, zero_division=0)
    })
display(pd.DataFrame(threshold_rows).round(4))


## 11. Conclusion — complete using the actual results above
After running all cells, record:
- Best model by held-out ROC-AUC and its measured score.
- Whether the >0.80 target was reached (use the printed output).
- Churn precision, recall, and F1 at threshold 0.50.
- Three strongest features from the feature-importance table.

### Limitations
- The IBM data is a public sample dataset, not live customer data.
- A random stratified split does not represent temporal forecasting.
- ROC-AUC does not assess probability calibration by itself.
- Feature importance does not establish causality.

### Submission checklist
- [ ] Run every cell from top to bottom without errors.
- [ ] Confirm the source loaded 7,043 rows and required columns.
- [ ] Confirm both Random Forest and XGBoost appear in the metrics table.
- [ ] Save the executed notebook with plots and metric outputs.
- [ ] Add the notebook to a public GitHub repository with a README.
- [ ] Share a LinkedIn walkthrough and tag Datheon as requested by the internship instructions.
